## Preprocessing
---
**Import Statements and Settings**

In [ ]:
# Resamples CT scans and masks to target standardized voxel dimensions 

In [ ]:
import os
import random
import numpy as np
from numpy import savez_compressed
from scipy.ndimage import zoom
import SimpleITK as sitk
from tqdm import tnrange, tqdm_notebook
from multiprocessing import Pool

In [ ]:
ct_dir = "" # LiTS CT folder (e.g., .../lits/ct)
mask_dir = "" # LiTS mask folder (e.g., .../lits/seg)
output_ct_dir = "" # where to save processed CT .npz
output_mask_dir = "" # where to save processed mask .npz

ct_prefix = "volume-" # LiTS CT prefix
mask_prefix = "segmentation-" # LiTS mask prefix

In [ ]:
def list_ct_files(directory):
    files = []
    for name in os.listdir(directory):
        if name.endswith(".nii") or name.endswith(".nii.gz"):
            files.append(name)
    return sorted(files)

ids = list_ct_files(ct_dir)

In [ ]:
# Select dimensions for CT/mask re-sampling 
resample_dims = (0.9765625*2.08,0.9765625*2.08,1.5*2.08) #low-resolution (for 192x160x80)
#resample_dims = (1,1,2) #high-resolution (for 192x192x96 model)
workers=38

In [ ]:
def split_nii_filename(filename):
    if filename.endswith(".nii.gz"):
        return filename[:-7], ".nii.gz"
    if filename.endswith(".nii"):
        return filename[:-4], ".nii"
    raise ValueError("Expected .nii or .nii.gz file: {name}".format(name=filename))


def preprocess(_id):
    print(_id)
    ct_stem, ct_ext = split_nii_filename(_id)
    mask_name = ct_stem.replace(ct_prefix, mask_prefix) + ct_ext

    ct_path = os.path.join(ct_dir, _id)
    mask_path = os.path.join(mask_dir, mask_name)

    ct = sitk.ReadImage(ct_path, sitk.sitkFloat32)
    mask = sitk.ReadImage(mask_path, sitk.sitkFloat32)

    ct_arr = sitk.GetArrayFromImage(ct)
    mask_arr = sitk.GetArrayFromImage(mask)

    ct_arr = np.moveaxis(ct_arr, 0, -1)
    mask_arr = np.moveaxis(mask_arr, 0, -1)

    ct_spacing = ct.GetSpacing()

    # Resample CT to specified target spacing
    zoom_factors = tuple([i / j for i, j in zip(ct_spacing, resample_dims)])
    ct_arr = zoom(ct_arr, zoom_factors, order=1)
    mask_arr = zoom(mask_arr, zoom_factors, order=0)
    mask_arr = np.round(mask_arr)

    os.makedirs(output_ct_dir, exist_ok=True)
    os.makedirs(output_mask_dir, exist_ok=True)

    ct_out = os.path.join(output_ct_dir, ct_stem + ".npz")
    mask_out = os.path.join(output_mask_dir, ct_stem + ".npz")

    savez_compressed(ct_out, ct_arr)
    savez_compressed(mask_out, mask_arr)


In [ ]:
def preprocess_mp(id_batch, workers=30):
    
 
    pool = Pool(processes=workers)

    pool.map(preprocess, id_batch)
    
    pool.close()
    

In [ ]:
id_batches = [ids[i * workers:(i + 1) * workers] for i in range((len(ids) + workers - 1) // workers )] # split into chunks of size = workers

for i in tnrange(len(id_batches)):
    id_batch = id_batches[i]
    
    preprocess_mp(id_batch, workers)